# Setup

In [ ]:
%run common.py
import sys
sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'
import networkit as nk
import multiprocessing

In [ ]:
def largest_connected_component(G):
    return G.subgraph(sorted(nx.connected_components(G.to_undirected()), key=lambda x: len(x))[-1])

In [ ]:
de_crossreference_path = f'../../legal-networks-data/de/4_crossreference_graph/seqitems'

In [ ]:
G_crossreference = nx.read_gpickle(f'{de_crossreference_path}/2019-01-01.gpickle.gz')

In [ ]:
counter_art = defaultdict(int)
counter_para= defaultdict(int)

for node, heading in nx.get_node_attributes(G_crossreference, 'heading').items():
    if  G_crossreference.nodes[node]['type'] == 'seqitem':
        abk = node.split('_')[1]
        if heading.startswith('§'):
            counter_para[abk] += 1
        elif heading.lower().startswith('art'):
            counter_art[abk] += 1
        
abk_units = {
    abk: ('§' if counter_para[abk] > counter_art[abk] else 'Art')
    for abk in (set(counter_art) | set(counter_para))
}

# Kookurrenzen von Gesetzeszitaten mit Referenzen auf andere Gesetze

In [ ]:
G = nx.read_gpickle('../../legal-networks-data/de_decisions/2_network.gpickle.gz')

In [ ]:
gerichte = sorted({g for n, g in G.nodes(data='gericht') if g})

In [ ]:
propagate_attrs_to_descendents(G, ['gericht', 'spruchkoerper', 'datum'])

In [ ]:
G_ms = quotient_decision_graph(G, merge_decisions=False, merge_statutes=True)

In [ ]:
H = make_occurrence_graph(G)
H_ms = make_occurrence_graph(G_ms)

In [ ]:
gerichte_filtered_Gs = [
    H.edge_subgraph((u,v,k) for u, v, k, g in H.edges(keys=True, data='gericht') if g == gericht)
    for gericht in gerichte
]

In [ ]:
gerichte_filtered_G_mss = [
    H_ms.edge_subgraph((u,v,k) for u, v, k, g in H_ms.edges(keys=True, data='gericht') if g == gericht)
    for gericht in gerichte
]

In [ ]:
cooccurrences_seqitem_to_document = []
for gericht, gerichte_filtered_G in zip(gerichte, gerichte_filtered_Gs):
    J = nx.Graph(gerichte_filtered_G)
    counts = sorted([
        (
            n, 
            (len({v.split('_')[0] for u, v in J.edges(n)}) - 1)
        )
        for n in J.nodes
    ], key=lambda x: -x[1])
    counts = counts[:50]
    counts = list(select_citekey(counts, abk_units))
    cooccurrences_seqitem_to_document.append(counts)
    print(f'{gericht} done ({gerichte.index(gericht)+1}/{len(gerichte)})')

In [ ]:
df = pd.DataFrame()
for gericht, counts in zip(gerichte, cooccurrences_seqitem_to_document):
    if gericht in ['BPatG', 'GmSOGB']:
        continue
    df_temp = pd.DataFrame([list(reversed(x)) for x in counts], columns=['#', 'Norm']).T
    df_temp['#'] = gericht
    df_temp = df_temp.set_index(['#', df_temp.index])
    df = df.append(df_temp, sort=False)
df = df.T
df = df.reset_index(drop=True)

tex = df.head(50).fillna('').to_latex(index=False).replace('BAG &', '\multicolumn{2}{l}{BAG} &').replace('\\toprule\n', '').replace('\\bottomrule', '').replace('\multicolumn{2}{l}', '\multicolumn{2}{c}')
tex = re.sub(r"-\d{4}", "...", tex)
print(tex)

with open(f'../tables/mikro_kokkurrenzen_top_andere_gesetze_de.tex', 'w') as f:
    f.write(tex)
    
df_gradz_anderes_gesetz = df

# Zentralitäten

In [ ]:
def get_network_stats(idx, G=None, mapping=None, nxG=None):
    gericht = gerichte[idx] if idx else ''
    if not G:
        G = gerichte_filtered_nkGs[idx]
    if not mapping:
        mapping = gerichte_filtered_nkG_mappings[idx]
    b_runner = nk.centrality.ApproxBetweenness(nk.graphtools.toUndirected(G), epsilon=0.1)
    b_runner.run()
    betweenness =  b_runner.ranking()[:50]
    betweenness = [
        (mapping[n], v)
        for n, v in betweenness
    ]
    print(gericht, 'betweenness done')
    c_runner = nk.centrality.TopCloseness(G, 50, first_heu=True)
    c_runner.run()
    closeness = c_runner.topkNodesList()
    closeness = [
        mapping[n]
        for n in closeness
    ]
    print(gericht, 'closeness done')


    if not nxG:
        nxG = weighted_Gs[idx]
    betweenness_flow_dict = nx.centrality.approximate_current_flow_betweenness_centrality(
        largest_connected_component(nxG), weight='weight', solver='lu'
    )
    betweenness_flow = sorted(betweenness_flow_dict.items(), key=lambda x: -x[-1])[:50]
    print(gericht, 'done')
    
    return betweenness, closeness, betweenness_flow

## Kookurrenzen in Gerichtsentscheidungen - Seqitems

In [ ]:
weighted_Gs = [
    make_weighted(G)
    for G in gerichte_filtered_Gs
]
gerichte_filtered_nkGs = [
    nk.nxadapter.nx2nk(G, weightAttr='weight')
    for G in weighted_Gs
]
gerichte_filtered_nkG_mappings = [
    {u: nid for (nid, u) in zip(G.nodes(), range(G.number_of_nodes()))}
    for G in weighted_Gs
]

In [ ]:
stats_seqitems = [
    get_network_stats(idx)
    for idx in range(len(gerichte_filtered_nkGs))
]

In [ ]:
df_gerichte = None
for idx in range(len(gerichte)):
    gericht = gerichte[idx]
    betweenness, closeness, betweenness_flow = stats_seqitems[idx]
    if gericht not in ['BPatG', 'GmSOGB']:
        betweenness = list(select_citekey(betweenness, abk_units))
        betweenness_flow = list(select_citekey(betweenness_flow, abk_units))
        df_gericht = pd.DataFrame(
            [[*b, *bf] for b, bf in zip(betweenness, betweenness_flow)],
            columns=['Zwischenz.', 'b_val', 'Zwischenz. (flussb.)', 'bf_val'])
        closeness = list(select_citekey([(c, None) for c in closeness], abk_units))
        df_gericht['Nähez.'] = [c for c, _ in closeness]
        df_gericht['Gradz. (anderes Gesetz)'] = df_gradz_anderes_gesetz[gericht]['Norm']
        df_gericht = df_gericht.drop('b_val', axis=1).drop('bf_val', axis=1).T
        df_gericht['Gericht'] = gerichte[idx]
        df_gericht.index = [df_gericht.Gericht, df_gericht.index]
        df_gericht = df_gericht.drop('Gericht', axis=1).T
        df_gerichte = df_gerichte.join(df_gericht) if df_gerichte is not None else df_gericht

In [ ]:
tex = ''
for gericht in gerichte:
    if gericht not in ['BPatG', 'GmSOGB']:
        tex += df_gerichte[gericht].iloc[:25].to_latex(index=False,).replace(
            '\\toprule', 
            '\\toprule\n\multicolumn{4}{l}{'+gericht+'} \\\\'
        )
        tex += '\n'
with open('../tables/mikro_centrality_cooccurences_seqitems_gerichte.tex', 'w') as f:
    f.write(tex)

## Kookurrenzen in Gerichtsentscheidungen - Seqitems

### Alle

In [ ]:
K_weighted = make_weighted(H_ms)
nkK = nk.nxadapter.nx2nk(K_weighted, weightAttr='weight')
mapping = {u: nid for (nid, u) in zip(K_weighted.nodes(), range(K_weighted.number_of_nodes()))}

In [ ]:
betweenness, closeness, betweenness_flow = get_network_stats(None, G=nkK, mapping=mapping, nxG=K_weighted.to_undirected())

In [ ]:
df = pd.DataFrame([(
    b_n, 
    c_n, 
    bf_n
) for (b_n), (c_n, _), (bf_n, _) in zip(
    closeness,
    betweenness,
    betweenness_flow,
)], columns=['Nähez.', 'Zwischenz.', 'Zwischenz. (flussb.)'])
df.head(25).to_latex('../tables/mikro_centrality_cooccurences_documents.tex', index=False,)
df

### Gerichtsspezifisch

In [ ]:
weighted_Gs = [
    make_weighted(G)
    for G in gerichte_filtered_G_mss
]
gerichte_filtered_nkGs = [
    nk.nxadapter.nx2nk(G, weightAttr='weight')
    for G in weighted_Gs
]
gerichte_filtered_nkG_mappings = [
    {u: nid for (nid, u) in zip(G.nodes(), range(G.number_of_nodes()))}
    for G in weighted_Gs
]

In [ ]:
stats_document = [
    get_network_stats(idx)
    for idx in range(len(gerichte_filtered_G_mss))
]

In [ ]:
df_gerichte = None
for idx in range(len(gerichte)):
    gericht = gerichte[idx]
    betweenness, closeness, betweenness_flow = stats_document[idx]
    if gericht not in ['BPatG', 'GmSOGB']:
        df_gericht = pd.DataFrame(
            [[*b, *bf] for b, bf in zip(betweenness, betweenness_flow)],
            columns=['Zwischenz.', 'b_val', 'Zwischenz. (flussb.)', 'bf_val'])
        df_gericht['Nähez.'] = closeness
        df_gericht = df_gericht.drop('b_val', axis=1).drop('bf_val', axis=1).T
        df_gericht['Gericht'] = gerichte[idx]
        df_gericht.index = [df_gericht.Gericht, df_gericht.index]
        df_gericht = df_gericht.drop('Gericht', axis=1).T
        df_gerichte = df_gerichte.join(df_gericht) if df_gerichte is not None else df_gericht

In [ ]:
tex = ''
for gericht in gerichte:
    if gericht not in ['BPatG', 'GmSOGB']:
        tex += df_gerichte[gericht].iloc[:25].to_latex(index=False,).replace(
            '\\toprule', 
            '\\toprule\n\multicolumn{4}{l}{'+gericht+'} \\\\'
        )
        tex += '\n'
with open('../tables/mikro_centrality_cooccurences_documents_gerichte.tex', 'w') as f:
    f.write(tex)

## Querverweise (ohne Sequenz)

### Seqitems

In [ ]:
K = nx.read_gpickle(
    '../../legal-networks-data/de/10_preprocessed_graph/2019-01-01_1-0_1-0_0.gpickle.gz'
)

In [ ]:
# comment to include sequence
K = K.edge_subgraph([(u,v,k) for u,v,k,d in K.edges(keys=True, data=True) if d['edge_type'] == 'reference'])

In [ ]:
K_weighted = make_weighted(K)
nkK = nk.nxadapter.nx2nk(K_weighted, weightAttr='weight')
mapping = {u: nid for (nid, u) in zip(K_weighted.nodes(), range(K_weighted.number_of_nodes()))}

In [ ]:
betweenness, closeness, betweenness_flow = get_network_stats(None, G=nkK, mapping=mapping, nxG=K_weighted.to_undirected())

In [ ]:
df = pd.DataFrame([(
    b_n, 
    c_n, 
    bf_n
) for (b_n, _), (c_n, _), (bf_n, _) in zip(
    select_human_readable_citekey([(c, None) for c in closeness], K),
    select_human_readable_citekey(betweenness, K),
    select_human_readable_citekey(betweenness_flow, K),
)], columns=['Nähez.', 'Zwischenz.', 'Zwischenz. (flussb.)'])
df.head(25).to_latex('../tables/mikro_centrality_crossreferences_seqitems.tex', index=False)
df.head(25)

### Documents Books

In [ ]:
K = nx.read_gpickle(
    '../../legal-networks-data/de/10_preprocessed_graph/2019-01-01_0-0_1-0_-1.gpickle.gz'
)

In [ ]:
K_weighted = make_weighted(K)
nkK = nk.nxadapter.nx2nk(K_weighted, weightAttr='weight')
mapping = {u: nid for (nid, u) in zip(K_weighted.nodes(), range(K_weighted.number_of_nodes()))}

In [ ]:
betweenness, closeness, betweenness_flow = get_network_stats(None, G=nkK, mapping=mapping, nxG=K_weighted.to_undirected())

In [ ]:
df = pd.DataFrame([(
    b_n, 
    c_n, 
    bf_n
) for (b_n, _), (c_n, _), (bf_n, _) in zip(
    select_key_parts([(c, None) for c in closeness], 1),
    select_key_parts(betweenness, 1),
    select_key_parts(betweenness_flow, 1),
)], columns=['Nähez.', 'Zwischenz.', 'Zwischenz. (flussb.)'])
df.head(25).to_latex('../tables/mikro_centrality_crossreferences_documents.tex', index=False)
df